# LeetCode 1018 — Binary Prefix Divisible By 5 bằng Python

**Link đề:** https://leetcode.com/problems/binary-prefix-divisible-by-5/
**Độ khó:** Easy · **Chủ đề:** Array, Bit Manipulation, Math (số học đồng dư)

```python
class Solution:
    def prefixesDivBy5(self, nums: list[int]) -> list[bool]:
        ...
```

## Mục tiêu học
1. Hiểu cách **đọc dần** một số nhị phân từ trái sang phải: `x_mới = 2·x_cũ + bit`.
2. Ôn công cụ: `int(s, 2)`, `<<`, `|`, `%`, `list.append`, list comprehension.
3. Viết baseline (dựng chuỗi tiền tố / giữ số lớn) và thấy vì sao nó chậm.
4. Nắm insight **đồng dư**: chỉ cần giữ `x % 5` thay vì cả số `x`.
5. Nhìn bài toán như một **máy trạng thái (automaton) 5 trạng thái**.
6. Trace, chứng minh, phân tích độ phức tạp, so sánh 4 cách, làm bài tập biến thể (chia hết cho `k` bất kỳ, đọc hệ 10).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — tiền tố nhị phân nào chia hết cho 5?

Cho mảng nhị phân `nums` (mỗi phần tử là `0` hoặc `1`). Gọi `x_i` là số có biểu diễn nhị phân là `nums[0..i]` (bit đầu là bit **có trọng số lớn nhất**).

Ví dụ `nums = [1,0,1]`: `x_0 = 1₂ = 1`, `x_1 = 10₂ = 2`, `x_2 = 101₂ = 5`.

Trả về mảng `answer` kiểu `bool`, `answer[i] = True` nếu `x_i` chia hết cho 5.

| `nums` | Các tiền tố | Giá trị | Output |
|---|---|---|---|
| `[0,1,1]` | `0`, `01`, `011` | 0, 1, 3 | `[True, False, False]` |
| `[1,1,1]` | `1`, `11`, `111` | 1, 3, 7 | `[False, False, False]` |
| `[1,0,1]` | `1`, `10`, `101` | 1, 2, 5 | `[False, False, True]` |

**Constraints:** `1 <= nums.length <= 10^5`, `nums[i]` là `0` hoặc `1`.

Câu hỏi tự đặt:
- `0` có chia hết cho 5 không? (Có: `0 % 5 == 0`.)
- Với `n = 10^5`, tiền tố cuối là số nhị phân **100 000 bit** — rất lớn. Có cần giữ cả số đó không?

In [1]:
nums = [1, 0, 1]
for i in range(len(nums)):
    prefix = nums[:i + 1]
    print(f"i={i}: tiền tố {prefix} -> chuỗi '{''.join(map(str, prefix))}'")

i=0: tiền tố [1] -> chuỗi '1'
i=1: tiền tố [1, 0] -> chuỗi '10'
i=2: tiền tố [1, 0, 1] -> chuỗi '101'


- `map(str, prefix)` áp `str` cho từng phần tử → `'1', '0', ...`.
- `''.join(...)` ghép các chuỗi con thành một chuỗi.

## 2. Nền tảng cần dùng

### 2.1. Đổi chuỗi nhị phân sang số: `int(s, 2)`

In [2]:
print(int("101", 2), int("011", 2), int("0", 2), int("111", 2))

5 3 0 7


### 2.2. Thêm một bit vào **bên phải**: `x_mới = 2·x + bit`

Trong hệ 10, viết thêm chữ số `d` vào bên phải số `x` cho `10·x + d` (ví dụ `12` → `123 = 12·10 + 3`).
Trong hệ 2 tương tự: thêm bit `b` vào bên phải `x` cho **`2·x + b`**. Đây chính là gợi ý của đề.

Cách viết bằng bit: `(x << 1) | b` — dịch trái 1 bit (nhân 2) rồi bật bit cuối nếu `b = 1`.

In [3]:
x = 0
for bit in [1, 0, 1, 1]:
    new_x = 2 * x + bit
    print(f"x = {x:>2} ({bin(x)[2:]:>4}), thêm bit {bit} -> 2*{x}+{bit} = {new_x:>2} ({bin(new_x)[2:]:>4}) | (x<<1)|bit = {(x << 1) | bit}")
    x = new_x

x =  0 (   0), thêm bit 1 -> 2*0+1 =  1 (   1) | (x<<1)|bit = 1
x =  1 (   1), thêm bit 0 -> 2*1+0 =  2 (  10) | (x<<1)|bit = 2
x =  2 (  10), thêm bit 1 -> 2*2+1 =  5 ( 101) | (x<<1)|bit = 5
x =  5 ( 101), thêm bit 1 -> 2*5+1 = 11 (1011) | (x<<1)|bit = 11


### 2.3. Phép `%` và số học đồng dư

`a % 5` là số dư khi chia `a` cho 5, luôn trong `0..4`. `a` chia hết cho 5 ⟺ `a % 5 == 0`.

**Tính chất then chốt:** phần dư của tổng/tích chỉ phụ thuộc phần dư của các thành phần:

```
(2·x + b) % 5  ==  (2·(x % 5) + b) % 5
```

Vì `x = 5q + r` → `2x + b = 10q + 2r + b`, mà `10q` chia hết cho 5.

In [4]:
for x in [7, 23, 1234567, 10**30 + 3]:
    for bit in [0, 1]:
        left = (2 * x + bit) % 5
        right = (2 * (x % 5) + bit) % 5
        assert left == right
        print(f"x={x}, bit={bit}: (2x+b)%5 = {left}, (2*(x%5)+b)%5 = {right}")

x=7, bit=0: (2x+b)%5 = 4, (2*(x%5)+b)%5 = 4
x=7, bit=1: (2x+b)%5 = 0, (2*(x%5)+b)%5 = 0
x=23, bit=0: (2x+b)%5 = 1, (2*(x%5)+b)%5 = 1
x=23, bit=1: (2x+b)%5 = 2, (2*(x%5)+b)%5 = 2
x=1234567, bit=0: (2x+b)%5 = 4, (2*(x%5)+b)%5 = 4
x=1234567, bit=1: (2x+b)%5 = 0, (2*(x%5)+b)%5 = 0
x=1000000000000000000000000000003, bit=0: (2x+b)%5 = 1, (2*(x%5)+b)%5 = 1
x=1000000000000000000000000000003, bit=1: (2x+b)%5 = 2, (2*(x%5)+b)%5 = 2


### 2.4. Xây list kết quả: `append` và list comprehension

In [5]:
answer = []
for v in [0, 1, 3, 5, 10]:
    answer.append(v % 5 == 0)
print(answer)
print([v % 5 == 0 for v in [0, 1, 3, 5, 10]])   # cùng kết quả, viết một dòng

[True, False, False, True, True]
[True, False, False, True, True]


`v % 5 == 0` là một **biểu thức so sánh**, cho ra `True`/`False` — ta `append` thẳng giá trị đó, không cần `if ... else`.

## 3. Cách trực giác / Baseline

### 3.1. Cách 1 — Dựng chuỗi tiền tố rồi `int(s, 2)` (O(n²))

In [6]:
def prefixes_div5_string(nums: list[int]) -> list[bool]:
    answer = []
    s = ""
    for bit in nums:
        s += str(bit)                    # chuỗi tiền tố dài thêm 1
        answer.append(int(s, 2) % 5 == 0)
    return answer

print(prefixes_div5_string([0, 1, 1]))
print(prefixes_div5_string([1, 1, 1]))
print(prefixes_div5_string([1, 0, 1]))

[True, False, False]
[False, False, False]
[False, False, True]


Mỗi lần gọi `int(s, 2)` tốn O(độ dài s) → tổng `1 + 2 + ... + n = O(n²)`. Với `n = 10^5` là khoảng `5·10^9` thao tác ký tự → quá chậm.

### 3.2. Cách 2 — Giữ số `x` đầy đủ, cập nhật `x = 2x + bit`

Không dựng lại chuỗi, chỉ cập nhật số. Python có **số nguyên không giới hạn** nên vẫn chạy đúng.

In [7]:
def prefixes_div5_bigint(nums: list[int]) -> list[bool]:
    answer = []
    x = 0
    for bit in nums:
        x = 2 * x + bit
        answer.append(x % 5 == 0)
    return answer

print(prefixes_div5_bigint([0, 1, 1]))
print(prefixes_div5_bigint([1, 0, 1]))

[True, False, False]
[False, False, True]


## 4. Nút thắt

Cách 2 có vẻ O(n) nhưng `x` dài thêm 1 bit mỗi vòng. Phép `2*x` và `x % 5` trên số `i` bit tốn O(i) → tổng vẫn **O(n²)** (dù hằng số nhỏ hơn cách chuỗi). Ở C++/Java còn tệ hơn: `x` tràn kiểu `int`/`long` sau 31/63 bit → **sai kết quả**.

Hãy đo thời gian để thấy rõ khi `n` tăng gấp đôi, thời gian tăng gần gấp 4 lần.

In [8]:
import time, random
random.seed(1018)

def measure(f, n):
    nums = [random.randint(0, 1) for _ in range(n)]
    start = time.perf_counter()
    f(nums)
    return time.perf_counter() - start

for n in [20_000, 40_000, 80_000]:
    print(f"n = {n:>6}: bigint {measure(prefixes_div5_bigint, n):.3f}s")

n =  20000: bigint 0.058s


n =  40000: bigint 0.240s


n =  80000: bigint 0.996s


(Thời gian cụ thể phụ thuộc máy, nhưng xu hướng tăng nhanh hơn tuyến tính là rõ ràng.)

Câu hỏi: để biết `x_i` có chia hết cho 5, ta có **cần** cả `x_i` không? → Không! Ta chỉ cần `x_i % 5`.

## 5. Insight — chỉ giữ phần dư

Từ mục 2.3: `x_mới % 5 = (2·(x_cũ % 5) + bit) % 5`.

Vậy thay vì giữ `x`, ta giữ `r = x % 5` (luôn trong `0..4`):

```python
r = (2 * r + bit) % 5
answer.append(r == 0)
```

Mỗi bước là phép toán trên số **≤ 9** → O(1). Tổng O(n).

In [9]:
nums = [1, 0, 1, 1, 0, 1]
x = 0
r = 0
for bit in nums:
    x = 2 * x + bit
    r = (2 * r + bit) % 5
    print(f"bit={bit} | x={x:>3} | x%5={x % 5} | r={r} | khớp: {x % 5 == r}")

bit=1 | x=  1 | x%5=1 | r=1 | khớp: True
bit=0 | x=  2 | x%5=2 | r=2 | khớp: True
bit=1 | x=  5 | x%5=0 | r=0 | khớp: True
bit=1 | x= 11 | x%5=1 | r=1 | khớp: True
bit=0 | x= 22 | x%5=2 | r=2 | khớp: True
bit=1 | x= 45 | x%5=0 | r=0 | khớp: True


## 6. Lời giải chính — mổ xẻ rồi ghép

```python
answer = []                      # Đoạn 1: list kết quả
r = 0                            # Đoạn 2: phần dư của tiền tố rỗng = 0
for bit in nums:                 # Đoạn 3: đọc từng bit từ trái sang
    r = (2 * r + bit) % 5        #   cập nhật phần dư
    answer.append(r == 0)        #   ghi kết quả cho tiền tố hiện tại
return answer                    # Đoạn 4
```

- Vì sao `r = 0` ban đầu? Tiền tố rỗng có giá trị 0. Thêm bit đầu tiên `b` cho `2·0 + b = b` ✓.
- Có thể viết `r = ((r << 1) | bit) % 5` — cùng nghĩa.

In [10]:
def prefixes_div5_mod(nums: list[int]) -> list[bool]:
    answer = []
    r = 0
    for bit in nums:
        r = (2 * r + bit) % 5
        answer.append(r == 0)
    return answer

for nums in [[0, 1, 1], [1, 1, 1], [1, 0, 1]]:
    print(nums, "->", prefixes_div5_mod(nums))

[0, 1, 1] -> [True, False, False]
[1, 1, 1] -> [False, False, False]
[1, 0, 1] -> [False, False, True]


### 6.1. Cách 4 — Máy trạng thái (bảng chuyển trạng thái)

Vì `r` chỉ có 5 giá trị và bit chỉ có 2 giá trị, ta **tính sẵn** bảng `NEXT[r][bit] = (2r + bit) % 5`. Khi chạy chỉ cần tra bảng — không cần phép `%` nữa.

Đây là một **DFA** (deterministic finite automaton — máy trạng thái hữu hạn): 5 trạng thái `0..4`, đọc từng bit để chuyển trạng thái; trạng thái `0` là "chấp nhận" (chia hết).

In [11]:
NEXT = [[(2 * r + bit) % 5 for bit in (0, 1)] for r in range(5)]
print("r | bit 0 -> | bit 1 ->")
for r in range(5):
    print(f"{r} |    {NEXT[r][0]}     |    {NEXT[r][1]}")

r | bit 0 -> | bit 1 ->
0 |    0     |    1
1 |    2     |    3
2 |    4     |    0
3 |    1     |    2
4 |    3     |    4


`[[... for bit in (0, 1)] for r in range(5)]` là **list comprehension lồng nhau**: vòng ngoài theo `r` tạo 5 hàng, vòng trong theo `bit` tạo 2 cột.

In [12]:
def prefixes_div5_dfa(nums: list[int]) -> list[bool]:
    state = 0
    answer = []
    for bit in nums:
        state = NEXT[state][bit]
        answer.append(state == 0)
    return answer

for nums in [[0, 1, 1], [1, 1, 1], [1, 0, 1]]:
    print(nums, "->", prefixes_div5_dfa(nums))

[0, 1, 1] -> [True, False, False]
[1, 1, 1] -> [False, False, False]
[1, 0, 1] -> [False, False, True]


## 7. TRACE từng bước

Ví dụ `nums = [1,1,0,0,1,0,1]`.

| i | bit | r trước | `2r + bit` | r sau `% 5` | x thật | answer[i] |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 0 | 1 | 0 | 1 | 1 | 1 | False |
| 1 | 1 | 1 | 3 | 3 | 3 | False |
| 2 | 0 | 3 | 6 | 1 | 6 | False |
| 3 | 0 | 1 | 2 | 2 | 12 | False |
| 4 | 1 | 2 | 5 | **0** | 25 | **True** |
| 5 | 0 | 0 | 0 | **0** | 50 | **True** |
| 6 | 1 | 0 | 1 | 1 | 101 | False |

Quan sát: khi `r = 0`, thêm bit `0` giữ `r = 0` (nhân 2 một số chia hết cho 5 vẫn chia hết).

In [13]:
def trace_mod(nums):
    r = 0
    x = 0
    answer = []
    print(f"{'i':>2} | bit | r trước | 2r+bit | r sau | x thật | chia hết?")
    for i, bit in enumerate(nums):
        before = r
        raw = 2 * r + bit
        r = raw % 5
        x = 2 * x + bit
        answer.append(r == 0)
        print(f"{i:>2} |  {bit}  |    {before}    |   {raw:>2}   |   {r}   | {x:>6} | {r == 0}")
    return answer

print(trace_mod([1, 1, 0, 0, 1, 0, 1]))

 i | bit | r trước | 2r+bit | r sau | x thật | chia hết?
 0 |  1  |    0    |    1   |   1   |      1 | False
 1 |  1  |    1    |    3   |   3   |      3 | False
 2 |  0  |    3    |    6   |   1   |      6 | False
 3 |  0  |    1    |    2   |   2   |     12 | False
 4 |  1  |    2    |    5   |   0   |     25 | True
 5 |  0  |    0    |    0   |   0   |     50 | True
 6 |  1  |    0    |    1   |   1   |    101 | False
[False, False, False, False, True, True, False]


### 7.1. Trace bằng máy trạng thái — xem đường đi qua các trạng thái

In [14]:
def trace_dfa(nums):
    state = 0
    path = [state]
    for bit in nums:
        state = NEXT[state][bit]
        path.append(state)
    print("bits :", "   ".join(map(str, nums)))
    print("path :", " -> ".join(map(str, path)))
    return path

trace_dfa([1, 1, 0, 0, 1, 0, 1])
trace_dfa([0, 1, 1])

bits : 1   1   0   0   1   0   1
path : 0 -> 1 -> 3 -> 1 -> 2 -> 0 -> 0 -> 1
bits : 0   1   1
path : 0 -> 0 -> 1 -> 3


[0, 0, 1, 3]

## 8. Code nộp LeetCode (Python 3)

In [15]:
class Solution:
    def prefixesDivBy5(self, nums: list[int]) -> list[bool]:
        answer = []
        r = 0
        for bit in nums:
            r = (2 * r + bit) % 5
            answer.append(r == 0)
        return answer

sol = Solution()
assert sol.prefixesDivBy5([0, 1, 1]) == [True, False, False]
assert sol.prefixesDivBy5([1, 1, 1]) == [False, False, False]
print("Solution PASS 2 ví dụ của đề")

Solution PASS 2 ví dụ của đề


Phiên bản dùng toán tử bit và máy trạng thái:

In [16]:
class SolutionBits:
    def prefixesDivBy5(self, nums: list[int]) -> list[bool]:
        answer = []
        r = 0
        for bit in nums:
            r = ((r << 1) | bit) % 5
            answer.append(r == 0)
        return answer


class SolutionDFA:
    NEXT = [[0, 1], [2, 3], [4, 0], [1, 2], [3, 4]]

    def prefixesDivBy5(self, nums: list[int]) -> list[bool]:
        state = 0
        answer = []
        for bit in nums:
            state = self.NEXT[state][bit]
            answer.append(state == 0)
        return answer

assert SolutionDFA.NEXT == NEXT
assert SolutionBits().prefixesDivBy5([1, 0, 1]) == [False, False, True]
assert SolutionDFA().prefixesDivBy5([1, 0, 1]) == [False, False, True]
print("SolutionBits, SolutionDFA PASS")

SolutionBits, SolutionDFA PASS


`NEXT` trong `SolutionDFA` là **thuộc tính của class** (class attribute) — dùng chung cho mọi đối tượng, truy cập qua `self.NEXT`. Ta kiểm tra nó khớp bảng đã tính ở mục 6.1.

## 9. Vì sao đúng?

**Bất biến:** sau khi xử lý `nums[0..i]`, biến `r` bằng `x_i % 5`.

- **Khởi đầu** (trước bit đầu): `r = 0 = x_{-1} % 5` với tiền tố rỗng có giá trị 0.
- **Duy trì:** giả sử `r = x_{i-1} % 5`. Ta có `x_i = 2·x_{i-1} + nums[i]`. Theo tính chất đồng dư (mục 2.3): `x_i % 5 = (2·(x_{i-1} % 5) + nums[i]) % 5 = (2r + nums[i]) % 5` — đúng giá trị `r` mới.
- **Kết luận:** `answer[i] = (r == 0) = (x_i % 5 == 0)` ⟺ `x_i` chia hết cho 5.

Không có chỗ nào phụ thuộc số 5 là số nguyên tố → kỹ thuật đúng cho mọi số chia `k ≥ 1`.

In [17]:
# Kiểm chứng bất biến r == x % 5 trên một dãy ngẫu nhiên dài
random.seed(42)
nums = [random.randint(0, 1) for _ in range(500)]
x = r = 0
for bit in nums:
    x = 2 * x + bit
    r = (2 * r + bit) % 5
    assert r == x % 5
print("Bất biến r == x % 5 đúng trên 500 bit")

Bất biến r == x % 5 đúng trên 500 bit


## 10. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm (ngoài output) |
|---|---|---|---|
| 1. Chuỗi + `int(s, 2)` | dựng lại số mỗi bước | `O(n²)` | `O(n)` (chuỗi) |
| 2. Số lớn `x = 2x + b` | giữ cả số | `O(n²)` (số học số lớn) | `O(n)` bit |
| 3. Phần dư `r` | đồng dư | `O(n)` | `O(1)` |
| 4. Máy trạng thái | tra bảng | `O(n)` | `O(1)` (bảng 5×2) |

Output `answer` luôn chiếm `O(n)` — đó là yêu cầu của đề, không tính vào bộ nhớ "thêm".

In [18]:
for n in [20_000, 40_000, 80_000]:
    print(f"n = {n:>6}: bigint {measure(prefixes_div5_bigint, n):.4f}s | mod {measure(prefixes_div5_mod, n):.4f}s")

n =  20000: bigint 0.0698s | mod 0.0016s


n =  40000: bigint 0.2770s | mod 0.0016s


n =  80000: bigint 0.9973s | mod 0.0034s


## 11. Kiểm thử: mẫu đề, biên, đối chiếu

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<36} PASS")

cases = [
    ("đề 1", [0, 1, 1], [True, False, False]),
    ("đề 2", [1, 1, 1], [False, False, False]),
    ("ví dụ trong đề [1,0,1]", [1, 0, 1], [False, False, True]),
    ("1 phần tử 0", [0], [True]),
    ("1 phần tử 1", [1], [False]),
    ("toàn 0", [0, 0, 0, 0], [True, True, True, True]),
    ("25 = 11001 rồi thêm 0 -> 50", [1, 1, 0, 0, 1, 0], [False, False, False, False, True, True]),
    ("10 = 1010", [1, 0, 1, 0], [False, False, True, True]),
]
for name, nums, exp in cases:
    check(name, Solution().prefixesDivBy5(nums), exp)

đề 1                                 PASS
đề 2                                 PASS
ví dụ trong đề [1,0,1]               PASS
1 phần tử 0                          PASS
1 phần tử 1                          PASS
toàn 0                               PASS
25 = 11001 rồi thêm 0 -> 50          PASS
10 = 1010                            PASS


### 11.1. Đối chiếu mọi cách (random có seed)

In [20]:
random.seed(2024)
solvers = {
    "string": prefixes_div5_string,
    "bigint": prefixes_div5_bigint,
    "mod": prefixes_div5_mod,
    "dfa": prefixes_div5_dfa,
    "Solution": Solution().prefixesDivBy5,
    "SolutionBits": SolutionBits().prefixesDivBy5,
    "SolutionDFA": SolutionDFA().prefixesDivBy5,
}
for _ in range(500):
    nums = [random.randint(0, 1) for _ in range(random.randint(1, 60))]
    expected = prefixes_div5_string(nums)
    for name, f in solvers.items():
        assert f(nums) == expected, (name, nums)
print("500 mảng ngẫu nhiên: mọi cách khớp nhau")

big = [random.randint(0, 1) for _ in range(100_000)]
res = Solution().prefixesDivBy5(big)
print("n = 10^5:", len(res), "kết quả, số tiền tố chia hết cho 5 =", sum(res))
assert res == SolutionDFA().prefixesDivBy5(big)

500 mảng ngẫu nhiên: mọi cách khớp nhau
n = 10^5: 100000 kết quả, số tiền tố chia hết cho 5 = 19824


`sum(res)` cộng các `True` như `1`, `False` như `0` → đếm số tiền tố chia hết cho 5. Với dữ liệu ngẫu nhiên, con số này gần `n / 5` vì phần dư phân bố khá đều.

## 12. Sai lầm người mới hay mắc

### 12.1. Đọc bit theo sai chiều (coi `nums[0]` là bit thấp nhất)

In [21]:
def wrong_lsb_first(nums):
    answer = []
    x = 0
    for i, bit in enumerate(nums):
        x += bit << i            # SAI: coi nums[i] có trọng số 2^i
        answer.append(x % 5 == 0)
    return answer

print("sai :", wrong_lsb_first([1, 0, 0, 1, 1]))
print("đúng:", Solution().prefixesDivBy5([1, 0, 0, 1, 1]))

sai : [False, False, False, False, True]
đúng: [False, False, False, False, False]


**Vì sao sai:** đề nói `nums[0]` là bit **có trọng số lớn nhất**. Với `[1,0,0,1,1]`, tiền tố đầy đủ là `10011₂ = 19` (không chia hết cho 5); code sai đọc ngược thành `11001₂ = 25` nên báo `True` sai. **Sửa:** dùng `x = 2x + bit` (dịch cái cũ sang trái, thêm bit mới vào bên phải).

### 12.2. Quên lấy `% 5` sau khi nhân (trong ngôn ngữ có giới hạn số nguyên)

Trong Python vẫn đúng nhưng chậm (O(n²)). Trong C++/Java với `int`, sau 31 bit `x` tràn → kết quả sai. Hãy mô phỏng tràn 32 bit:

In [22]:
def wrong_overflow_32(nums):
    answer = []
    x = 0
    for bit in nums:
        x = (2 * x + bit) & 0xFFFFFFFF      # mô phỏng unsigned 32-bit: chỉ giữ 32 bit thấp
        answer.append(x % 5 == 0)
    return answer

random.seed(3)
nums = [random.randint(0, 1) for _ in range(100)]
bad = wrong_overflow_32(nums)
good = Solution().prefixesDivBy5(nums)
first_diff = next(i for i in range(len(nums)) if bad[i] != good[i])
print("Kết quả lệch lần đầu tại i =", first_diff, "(sau khi x vượt 32 bit)")

Kết quả lệch lần đầu tại i = 34 (sau khi x vượt 32 bit)


`0xFFFFFFFF` là số hex = `2^32 - 1` (32 bit 1). `x & 0xFFFFFFFF` giữ 32 bit thấp — giống hiện tượng tràn số. `next(...)` lấy phần tử đầu tiên của generator.

**Sửa:** luôn `% 5` mỗi bước.

### 12.3. Chỉ lấy `% 5` cho phần nhân, quên phần cộng bit

In [23]:
def wrong_order(nums):
    answer = []
    r = 0
    for bit in nums:
        r = (2 * r) % 5 + bit      # SAI: r có thể bằng 5
        answer.append(r == 0)
    return answer

print("sai :", wrong_order([1, 0, 1]))
print("đúng:", Solution().prefixesDivBy5([1, 0, 1]))

sai : [False, False, False]
đúng: [False, False, True]


**Vì sao sai:** với `[1,0,1]`: `r = 1`, rồi `2`, rồi `(4) % 5 + 1 = 5`. Giá trị thật là 5 (chia hết), nhưng `r = 5 ≠ 0` nên code báo `False`. `r` phải luôn nằm trong `0..4`. **Sửa:** lấy `%` **sau cùng**: `r = (2 * r + bit) % 5`.

### 12.4. Trả về số thay vì `bool`

`answer.append(r)` hoặc `answer.append(int(r == 0))` → LeetCode yêu cầu `list[bool]`. Python coi `1 == True` nên `assert` có thể vẫn qua, nhưng nên trả đúng kiểu `True/False`.

In [24]:
print(1 == True, [1, 0] == [True, False], type(1), type(True))

True True <class 'int'> <class 'bool'>


## 13. Nhận diện pattern

Dấu hiệu:
- "số tạo từ tiền tố các chữ số/bit, kiểm tra chia hết" → giữ **phần dư** và cập nhật `r = (base·r + digit) % k`.
- "số rất lớn, chỉ hỏi chia hết/đồng dư" → không bao giờ giữ cả số.
- "mỗi bước chỉ phụ thuộc trạng thái nhỏ" → nghĩ tới **máy trạng thái**.

Bài liên quan:
- LeetCode 1009 (Complement of Base 10 Integer) — bit cơ bản.
- LeetCode 1290 (Convert Binary Number in a Linked List to Integer) — cùng công thức `x = 2x + bit`.
- LeetCode 2575 (Find the Divisibility Array of a String) — giống hệt nhưng hệ 10, chia cho `m`.
- LeetCode 1015 (Smallest Integer Divisible by K) — chuỗi `1, 11, 111...` modulo `k`.

## 14. Biến thể

### 14.1. Biến thể A — chia hết cho `k` bất kỳ, cơ số `base` bất kỳ

In [25]:
def prefixes_div_k(digits: list[int], k: int, base: int = 2) -> list[bool]:
    answer = []
    r = 0
    for d in digits:
        r = (base * r + d) % k
        answer.append(r == 0)
    return answer

print(prefixes_div_k([1, 0, 1, 1], 3))              # 1, 2, 5, 11 -> chia hết 3?
print(prefixes_div_k([9, 9, 8, 2], 3, base=10))     # 9, 99, 998, 9982 -> chia hết 3?

[False, False, False, False]
[True, True, False, False]


### 14.2. Biến thể B — đếm số tiền tố chia hết cho 5 (chỉ cần một số)

In [26]:
def count_prefixes_div5(nums: list[int]) -> int:
    r = 0
    count = 0
    for bit in nums:
        r = (2 * r + bit) % 5
        count += r == 0
    return count

print(count_prefixes_div5([1, 1, 0, 0, 1, 0, 1]))   # 2 (25 và 50)

2


`count += r == 0` cộng `True` (1) hoặc `False` (0). Toán tử `==` có độ ưu tiên cao hơn `+=` nên không cần ngoặc (nhưng viết `count += (r == 0)` dễ đọc hơn).

## 15. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Với `nums = [1, 0, 0, 1, 1, 0]`, lập bảng `r` như mục 7 và cho mảng kết quả.

### Bài 2 — Dự đoán output
Đoán `NEXT[3][1]`, `NEXT[4][0]` và `Solution().prefixesDivBy5([1, 0, 1, 0, 0])` trước khi chạy.

### Bài 3 — TODO: viết lại lời giải phần dư

In [27]:
def exercise_prefixes_div5(nums: list[int]) -> list[bool]:
    # TODO 1: r = 0, answer = []
    # TODO 2: với mỗi bit: r = (2*r + bit) % 5; answer.append(r == 0)
    pass

# assert exercise_prefixes_div5([0, 1, 1]) == [True, False, False]
# assert exercise_prefixes_div5([1, 0, 1]) == [False, False, True]

### Bài 4 — TODO: chuỗi thập phân, chia hết cho `m` (LeetCode 2575)

Cho chuỗi chữ số `word` và số `m`, trả về list `div` với `div[i] = 1` nếu tiền tố `word[0..i]` chia hết cho `m`, ngược lại `0`.

In [28]:
def exercise_divisibility_array(word: str, m: int) -> list[int]:
    # TODO
    pass

# assert exercise_divisibility_array("998244353", 3) == [1, 1, 0, 0, 0, 1, 1, 0, 0]
# assert exercise_divisibility_array("1010", 10) == [0, 1, 0, 1]

### Bài 5 — TODO biến thể: bit đi **ngược chiều** (bit đầu là bit thấp nhất)

`x_i = nums[0]·2^0 + nums[1]·2^1 + ... + nums[i]·2^i`. Trả về `x_i % 5 == 0` cho mỗi `i`, trong O(n) và O(1) bộ nhớ thêm.

In [29]:
def exercise_lsb_first_div5(nums: list[int]) -> list[bool]:
    # TODO: giữ r = x % 5 và p = 2^i % 5
    pass

# assert exercise_lsb_first_div5([1, 0, 1, 0]) == [False, False, True, True]
# assert exercise_lsb_first_div5([0]) == [True]

### Bài 6 — Mini challenge: xây bảng máy trạng thái cho số chia `k` bất kỳ và chạy nó

Viết `exercise_build_dfa(k)` trả về bảng `NEXT` kích thước `k × 2`, rồi dùng bảng đó để kiểm tra mọi tiền tố của một dãy bit.

In [30]:
def exercise_build_dfa(k: int) -> list[list[int]]:
    # TODO
    pass

# assert exercise_build_dfa(5) == [[0, 1], [2, 3], [4, 0], [1, 2], [3, 4]]
# assert exercise_build_dfa(3) == [[0, 1], [2, 0], [1, 2]]

## 16. Gợi ý ba mức

**Bài 1** — (1) `r` bắt đầu 0; (2) `r = (2r + bit) % 5`; (3) các giá trị thật: 1, 2, 4, 9, 19, 38.

**Bài 2** — (1) `NEXT[r][b] = (2r + b) % 5`; (2) `2·3+1 = 7`, `2·4+0 = 8`; (3) tiền tố `1, 10, 101, 1010, 10100` = 1, 2, 5, 10, 20.

**Bài 3** — (1) mục 6; (2) `for bit in nums`; (3) bốn dòng code.

**Bài 4** — (1) cơ số 10; (2) `int(ch)` đổi ký tự thành số; (3) `r = (10 * r + int(ch)) % m`, `append(1 if r == 0 else 0)`.

**Bài 5** — (1) bit thứ `i` đóng góp `bit · 2^i`; (2) giữ `p = 2^i % 5`, mỗi bước `p = 2p % 5`; (3) `r = (r + bit * p) % 5`.

**Bài 6** — (1) list comprehension lồng; (2) `range(k)` và `(0, 1)`; (3) `[[(2*r + b) % k for b in (0, 1)] for r in range(k)]`.

## 17. Đáp án tham khảo

**Bài 1:** `r`: 1, 2, 4, 4 (9%5), 4 (19%5), 3 (38%5) → `[False]*6`.

**Bài 2:** `NEXT[3][1] = 2`, `NEXT[4][0] = 3`, kết quả `[False, False, True, True, True]` (5, 10, 20 chia hết cho 5).

In [31]:
def answer_prefixes_div5(nums):
    r = 0
    answer = []
    for bit in nums:
        r = (2 * r + bit) % 5
        answer.append(r == 0)
    return answer


def answer_divisibility_array(word, m):
    r = 0
    div = []
    for ch in word:
        r = (10 * r + int(ch)) % m
        div.append(1 if r == 0 else 0)
    return div


def answer_lsb_first_div5(nums):
    r = 0
    p = 1                       # 2^0 % 5
    answer = []
    for bit in nums:
        r = (r + bit * p) % 5
        answer.append(r == 0)
        p = (2 * p) % 5
    return answer


def answer_build_dfa(k):
    return [[(2 * r + b) % k for b in (0, 1)] for r in range(k)]


assert answer_prefixes_div5([1, 0, 0, 1, 1, 0]) == [False] * 6
assert (NEXT[3][1], NEXT[4][0]) == (2, 3)
assert Solution().prefixesDivBy5([1, 0, 1, 0, 0]) == [False, False, True, True, True]
assert answer_prefixes_div5([0, 1, 1]) == [True, False, False]
assert answer_divisibility_array("998244353", 3) == [1, 1, 0, 0, 0, 1, 1, 0, 0]
assert answer_divisibility_array("1010", 10) == [0, 1, 0, 1]
assert answer_lsb_first_div5([1, 0, 1, 0]) == [False, False, True, True]
assert answer_lsb_first_div5([0]) == [True]
assert answer_build_dfa(5) == [[0, 1], [2, 3], [4, 0], [1, 2], [3, 4]]
assert answer_build_dfa(3) == [[0, 1], [2, 0], [1, 2]]

random.seed(11)
for _ in range(300):
    nums = [random.randint(0, 1) for _ in range(random.randint(1, 40))]
    xs = [sum(nums[j] << j for j in range(i + 1)) for i in range(len(nums))]
    assert answer_lsb_first_div5(nums) == [x % 5 == 0 for x in xs]
    for k in (3, 7, 10):
        table = answer_build_dfa(k)
        st = 0
        for i, bit in enumerate(nums):
            st = table[st][bit]
            assert st == int("".join(map(str, nums[:i + 1])), 2) % k
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 5:** khi bit đầu là bit thấp, bit mới đi vào **bên trái** với trọng số `2^i` lớn dần. Ta không thể dùng `2r + bit`; thay vào đó giữ `p = 2^i mod 5` (chu kỳ `1, 2, 4, 3, 1, ...`) và cộng `bit·p`.

## 18. Tổng kết — template + checklist

**Template nhớ nhanh:**

```python
class Solution:
    def prefixesDivBy5(self, nums: list[int]) -> list[bool]:
        answer = []
        r = 0
        for bit in nums:
            r = (2 * r + bit) % 5
            answer.append(r == 0)
        return answer
```

**Câu thần chú:** *"Thêm bit bên phải = nhân 2 cộng bit — chỉ giữ phần dư."*

**Checklist trước khi submit:**
- [ ] Đọc bit từ trái sang phải, `nums[0]` là bit cao nhất?
- [ ] Công thức `2 * r + bit` (không phải `2 * (r + bit)`)?
- [ ] `% 5` **mỗi bước** (không giữ số lớn)?
- [ ] `r` khởi tạo bằng 0?
- [ ] Trả `list[bool]`, độ dài bằng `len(nums)`?
- [ ] Đã test `[0]`, toàn 0, ca có 25 → 50?